# Parte 2 · Exploración de registros del PDB
**Familia:** fosforribosilantranilato isomerasa (PRAI) · **Pfam:** `PF00697`  
**Equipo:** Mónica Reyes Ramírez, Natalie Pineda Morán y Rubí Chavarría Martínez  
**Fuente:** RCSB PDB Search API y Data API

## Alcance y criterios

Se analizan **entradas PDB únicas** que contienen al menos una entidad proteica anotada con Pfam `PF00697`, siguiendo el criterio de la Parte 1. El listado inicial proviene de `ids_final` (si ya fue generado en el mismo notebook), de `ids_familia.txt` o, si no existen, de una consulta nueva a la Search API. El número puede variar con las actualizaciones del PDB. En la ejecución de la Parte 1 del equipo se reportaron 11 entradas por Pfam** frente a 34 por búsqueda textual**, de las cuales 23 no coincidían con la selección de Pfam.

Se usa un identificador PDB para contabilizar una estructura, aunque haya más de una cadena. Las características de longitud y organismo se consultan únicamente para las entidades proteicas que tienen la anotación PF00697, evitando confundir proteínas acompañantes y el organismo de expresión con la especie de origen. En caso de varias entidades PRAI en una entrada, cada entidad se considera en el análisis de longitudes; para organismos se cuenta cada combinación entrada-organismo una sola vez. Un mismo identificador PDB puede aparecer en más de una categoría si contiene varios organismos o métodos. Los ligandos se cuentan por **número de entradas PDB donde aparecen**, no por número de moléculas.

**Reproducibilidad:** ejecutar todas las celdas en orden con conexión a Internet. La información se extrae en tiempo de ejecución; las gráficas y los CSV se guardan en `resultados_parte2/`.

## 0. Preparación

In [ ]:
# Si hace falta, instalar una sola vez desde Jupyter:
# %pip install requests pandas matplotlib

import re
from pathlib import Path
from collections import defaultdict
import requests
import pandas as pd
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

PFAM_ID = "PF00697"
BUSQUEDA_API = "https://search.rcsb.org/rcsbsearch/v2/query"
DATOS_API = "https://data.rcsb.org/rest/v1/core"
SALIDA = Path("resultados_parte2")
FIGURAS = SALIDA / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

session = requests.Session()
reintentos = Retry(total=3, backoff_factor=0.7,
                   status_forcelist=[429, 500, 502, 503, 504],
                   allowed_methods=["GET", "POST"])
session.mount("https://", HTTPAdapter(max_retries=reintentos))
session.headers.update({"Accept": "application/json"})

def get_rcsb(ruta):
    """Obtiene un registro de la Data API. Falla de forma explícita si no existe."""
    r = session.get(f"{DATOS_API}/{ruta}", timeout=30)
    r.raise_for_status()
    return r.json()

def guardar_figura(nombre):
    plt.tight_layout()
    plt.savefig(FIGURAS / nombre, dpi=180, bbox_inches="tight")
    plt.show()

## 1. Recuperar los IDs seleccionados en la Parte 1

Por prioridad, se reutiliza la variable `ids_final` de Mónica. Para ejecutar este notebook de forma independiente, puede leerse el archivo con un ID por línea o repetirse la búsqueda. Se imprimen los IDs para que el equipo pueda corroborarlos.

In [ ]:
if "ids_final" in globals() and ids_final:
    ids = sorted({str(x).strip().upper() for x in ids_final if str(x).strip()})
    procedencia_ids = "variable ids_final de la Parte 1"
elif Path("ids_familia.txt").exists():
    ids = sorted({x.strip().upper() for x in Path("ids_familia.txt").read_text().splitlines() if x.strip()})
    procedencia_ids = "archivo ids_familia.txt"
else:
    consulta = {
        "type": "group", "logical_operator": "and", "nodes": [
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_polymer_entity_annotation.type",
                "operator": "exact_match", "value": "Pfam"}},
            {"type": "terminal", "service": "text", "parameters": {
                "attribute": "rcsb_polymer_entity_annotation.annotation_id",
                "operator": "exact_match", "value": PFAM_ID}},
        ],
    }
    r = session.post(BUSQUEDA_API, json={
        "query": consulta, "return_type": "entry",
        "request_options": {"return_all_hits": True}}, timeout=60)
    if r.status_code == 204:
        ids = []
    else:
        r.raise_for_status()
        ids = sorted({h["identifier"].upper() for h in r.json().get("result_set", [])})
    procedencia_ids = "consulta nueva a la Search API"

print("Origen:", procedencia_ids)
print("Número de IDs iniciales:", len(ids))
print("Identificadores:", ", ".join(ids))
assert ids, "No se recuperaron IDs. Revisar el archivo o la consulta de Pfam."

## 2. Recuperar y depurar metadatos con la Data API

Se consultan las entradas (`entry`), sus entidades proteicas (`polymer_entity`) y sus moléculas pequeñas (`nonpolymer_entity`). Se documentan por separado entradas, entidades PRAI, organismos y ligandos. No se inventan valores cuando un campo no está disponible.
**Control de calidad:** se utiliza `rcsb_entry_container_identifiers.non_polymer_entity_ids` y se comprueba el sulfato SO4 de 1DL3 para detectar extracciones incompletas.


In [ ]:
def tiene_pfam(entidad, pfam_id=PFAM_ID):
    for anot in entidad.get("rcsb_polymer_entity_annotation", []) or []:
        if (str(anot.get("type", "")).lower() == "pfam"
                and str(anot.get("annotation_id", "")).upper() == pfam_id):
            return True
    return False

def organismos_de_origen(entidad):
    """No utilizar organismo de expresión para responder la pregunta 3."""
    organismos = set()
    for x in entidad.get("rcsb_entity_source_organism", []) or []:
        nombre = x.get("ncbi_scientific_name") or x.get("scientific_name")
        if nombre:
            organismos.add(nombre)
    # Campos originales como alternativa cuando no existe taxonomía normalizada
    if not organismos:
        for campo, atributo in [("entity_src_gen", "pdbx_gene_src_scientific_name"),
                                ("entity_src_nat", "pdbx_organism_scientific")]:
            valores = entidad.get(campo, []) or []
            if isinstance(valores, dict):
                valores = [valores]
            for x in valores:
                if x.get(atributo):
                    organismos.add(x[atributo])
    return organismos or {"No reportado"}

def longitud_secuencia(entidad):
    """Número de letras de la secuencia canónica de una entidad proteica."""
    seq = (entidad.get("entity_poly") or {}).get("pdbx_seq_one_letter_code_can")
    if not seq:
        return None
    secuencia = re.sub(r"\s+", "", seq)
    return len(secuencia)

def resolver_ligando(datos):
    info = datos.get("pdbx_entity_nonpoly") or {}
    info_comp = (datos.get("nonpolymer_comp") or {}).get("chem_comp") or {}
    codigo = info.get("comp_id") or info_comp.get("id")
    nombre = info.get("name") or info_comp.get("name") or "Nombre no reportado"
    return codigo, nombre

filas_entradas = []
filas_entidades = []
filas_organismos = []
filas_ligandos = []
incidencias = []

for pdb_id in ids:
    try:
        registro = get_rcsb(f"entry/{pdb_id}")
        indices = registro.get("rcsb_entry_container_identifiers", {})
        elegibles = []
        for entity_id in indices.get("polymer_entity_ids", []) or []:
            entidad = get_rcsb(f"polymer_entity/{pdb_id}/{entity_id}")
            if tiene_pfam(entidad):
                elegibles.append((str(entity_id), entidad))
        if not elegibles:
            incidencias.append(f"{pdb_id}: no se identificó entidad con anotación {PFAM_ID}; se excluye")
            continue
        metodos = sorted({x["method"] for x in registro.get("exptl", []) or []
                          if x.get("method")}) or ["No reportado"]
        lista_res = (registro.get("rcsb_entry_info") or {}).get("resolution_combined") or []
        resolucion = next((float(r) for r in lista_res if r is not None), None)
        fecha = (registro.get("rcsb_accession_info") or {}).get("deposit_date")
        filas_entradas.append({"pdb_id": pdb_id, "metodos": metodos,
                               "resolucion_A": resolucion, "fecha_deposito": fecha})
        organismos_entrada = set()
        for entity_id, entidad in elegibles:
            longitud = longitud_secuencia(entidad)
            organismos = organismos_de_origen(entidad)
            filas_entidades.append({"pdb_id": pdb_id, "entity_id": entity_id,
                                    "longitud_aa": longitud,
                                    "organismos": "; ".join(sorted(organismos))})
            organismos_entrada.update(organismos)
        for organismo in organismos_entrada:
            filas_organismos.append({"pdb_id": pdb_id, "organismo": organismo})
        vistos = set()
        for nonpolymer_id in indices.get("non_polymer_entity_ids", []) or []:
            datos_ligando = get_rcsb(f"nonpolymer_entity/{pdb_id}/{nonpolymer_id}")
            codigo, nombre = resolver_ligando(datos_ligando)
            if codigo and codigo not in {"HOH", "DOD"} and codigo not in vistos:
                filas_ligandos.append({"pdb_id": pdb_id, "ligando_id": codigo,
                                       "nombre": nombre})
                vistos.add(codigo)
    except requests.RequestException as e:
        incidencias.append(f"{pdb_id}: error de API: {e}")

entradas = pd.DataFrame(filas_entradas, columns=["pdb_id", "metodos", "resolucion_A", "fecha_deposito"])
entidades = pd.DataFrame(filas_entidades, columns=["pdb_id", "entity_id", "longitud_aa", "organismos"])
organismos = pd.DataFrame(filas_organismos, columns=["pdb_id", "organismo"]).drop_duplicates()
ligandos = pd.DataFrame(filas_ligandos, columns=["pdb_id", "ligando_id", "nombre"]).drop_duplicates()
entradas["fecha_deposito"] = pd.to_datetime(entradas["fecha_deposito"], errors="coerce")
entradas["anio"] = entradas["fecha_deposito"].dt.year

print(f"Entradas iniciales: {len(ids)} | analizables: {len(entradas)}")
print(f"Entidades anotadas como PRAI: {len(entidades)}")
print(f"Entradas con ligandos: {ligandos['pdb_id'].nunique()}")
if incidencias:
    print("INCIDENCIAS: revisar antes de interpretar resultados")
    for incidencia in incidencias:
        print(" -", incidencia)
assert not entradas.empty, "No se recuperaron entradas utilizables. Comprobar conectividad y anotaciones."
assert len(entradas) == len(ids), "No se recuperaron todos los registros; revisar incidencias de API."
# Control comprobable: 1DL3 contiene sulfato según el registro oficial RCSB.
if "1DL3" in ids:
    tiene_sulfato = ((ligandos["pdb_id"] == "1DL3") &
                    (ligandos["ligando_id"] == "SO4")).any()
    assert tiene_sulfato, "Falta SO4 en 1DL3: revisar extracción de ligandos." 
display(entradas.head())
display(entidades.head())

## Pregunta 1. ¿Cuántas estructuras se identificaron?

El denominador es el número de **entradas PDB únicas** que contienen una entidad PRAI confirmada. No se cuentan cadenas, copias cristalográficas ni ligandos como estructuras adicionales.

In [ ]:
numero_estructuras = entradas["pdb_id"].nunique()
print(f"Se identificaron {numero_estructuras} estructuras PDB que cumplen el criterio {PFAM_ID}.")

## Pregunta 2. Métodos experimentales

In [ ]:
metodos_tabla = entradas[["pdb_id", "metodos"]].explode("metodos").rename(columns={"metodos": "metodo"})
conteo_metodos = metodos_tabla.groupby("metodo")["pdb_id"].nunique().sort_values(ascending=False)
display(conteo_metodos.rename("n_estructuras").to_frame())
plt.figure(figsize=(8, max(3.5, 0.6 * len(conteo_metodos))))
conteo_metodos.sort_values().plot(kind="barh")
plt.title("Métodos experimentales de las estructuras PRAI")
plt.xlabel("Número de estructuras PDB")
plt.ylabel("Método experimental")
guardar_figura("02_metodos.png")

## Pregunta 3. Organismos de origen

Se usa la taxonomía de **origen de la entidad proteica PRAI**, no el hospedero de expresión. Una misma estructura puede contribuir a más de un organismo si presenta entidades PRAI de orígenes distintos.

In [ ]:
conteo_organismos = organismos.groupby("organismo")["pdb_id"].nunique().sort_values(ascending=False)
display(conteo_organismos.rename("n_estructuras").to_frame())
plt.figure(figsize=(9, max(4, 0.5 * min(10, len(conteo_organismos)))))
conteo_organismos.head(10).sort_values().plot(kind="barh")
plt.title("Organismos de origen más frecuentes (PRAI)")
plt.xlabel("Número de estructuras PDB")
plt.ylabel("Organismo")
guardar_figura("03_organismos.png")

## Pregunta 4. Resolución de estructuras cristalográficas

Se analizan únicamente las entradas cuyo método incluye `X-RAY DIFFRACTION`, con resolución disponible. La unidad es el ångström (Å); a menor valor, mayor detalle espacial nominal.

In [ ]:
ids_rayos_x = set(metodos_tabla.loc[
    metodos_tabla["metodo"].eq("X-RAY DIFFRACTION"), "pdb_id"])
rx = entradas.loc[entradas["pdb_id"].isin(ids_rayos_x), "resolucion_A"].dropna()
print(f"Estructuras por rayos X: {len(ids_rayos_x)}; con resolución numérica: {len(rx)}")
if len(rx):
    display(rx.describe().rename("resolucion_A").to_frame())
    plt.figure(figsize=(7, 4))
    plt.hist(rx, bins=min(8, max(1, len(rx))), edgecolor="black")
    plt.title("Distribución de resolución por difracción de rayos X")
    plt.xlabel("Resolución (Å)")
    plt.ylabel("Número de estructuras PDB")
    guardar_figura("04_resolucion_rayos_x.png")
else:
    print("No hay resoluciones de rayos X disponibles para graficar.")

## Pregunta 5. Línea de tiempo por fecha de depósito

Se usa la **fecha de depósito**, no la de publicación o liberación.

In [ ]:
por_anio = entradas.dropna(subset=["anio"]).groupby("anio")["pdb_id"].nunique().sort_index()
if not por_anio.empty:
    por_anio = por_anio.reindex(range(int(por_anio.index.min()), int(por_anio.index.max()) + 1), fill_value=0)
display(por_anio.rename("n_estructuras").to_frame())
if not por_anio.empty:
    plt.figure(figsize=(9, 4))
    plt.plot(por_anio.index, por_anio.values, marker="o")
    plt.title("Depósitos anuales de estructuras PRAI en el PDB")
    plt.xlabel("Año de depósito")
    plt.ylabel("Número de estructuras PDB")
    plt.xticks(por_anio.index, rotation=45)
    plt.grid(axis="y", alpha=0.3)
    guardar_figura("05_linea_tiempo.png")
else:
    print("No hay fechas de depósito disponibles.")

## Pregunta 6. Longitud de las proteínas

La longitud se define como el **número de aminoácidos de la secuencia canónica** (`entity_poly.pdbx_seq_one_letter_code_can`) para cada **entidad proteica anotada con PF00697**. No se multiplican los residuos por número de cadenas. En `1DL3`, por ejemplo, una cadena mide 203 aminoácidos; los 385 residuos modelados reportados para la entrada son un total agregado y no la longitud de una cadena.

In [ ]:
longitudes = pd.to_numeric(entidades["longitud_aa"], errors="coerce").dropna()
print(f"Entidades PRAI con longitud calculable: {len(longitudes)} de {len(entidades)}")
if len(longitudes):
    display(longitudes.describe().rename("longitud_aa").to_frame())
    plt.figure(figsize=(7, 4))
    plt.hist(longitudes, bins=min(8, max(1, len(longitudes))), edgecolor="black")
    plt.title("Distribución de longitudes de entidades proteicas PRAI")
    plt.xlabel("Longitud de la secuencia (aminoácidos)")
    plt.ylabel("Número de entidades proteicas")
    guardar_figura("06_longitudes.png")
else:
    print("No se pudo calcular ninguna longitud.")

## Pregunta 7. Cinco ligandos más frecuentes

Se contabiliza el número de **estructuras PDB distintas** que contienen cada ligando. Un mismo ligando repetido varias veces en una estructura solo aporta uno al conteo. Se excluye agua (`HOH`, `DOD`), pero se **conservan iones y componentes de cristalización** como `SO4`, que no necesariamente son ligandos funcionales.

In [ ]:
frecuencia_ligandos = (ligandos.groupby(["ligando_id", "nombre"])["pdb_id"]
                      .nunique().sort_values(ascending=False).head(5))
display(frecuencia_ligandos.rename("n_estructuras").to_frame())
if not frecuencia_ligandos.empty:
    etiquetas = [f"{codigo} — {nombre}" for codigo, nombre in frecuencia_ligandos.index]
    plt.figure(figsize=(10, max(3.5, 0.65 * len(etiquetas))))
    plt.barh(etiquetas[::-1], frecuencia_ligandos.values[::-1])
    plt.title("Cinco ligandos más frecuentes en estructuras PRAI")
    plt.xlabel("Número de estructuras PDB con el ligando")
    plt.ylabel("Ligando (ID — nombre)")
    guardar_figura("07_ligandos.png")
else:
    print("No se encontraron ligandos no acuosos entre las estructuras recuperadas.")

## Parte 4. Interpretación breve y limitaciones

Las respuestas siguientes se preparan automáticamente en función de los datos recuperados. Revisar la coherencia con las gráficas antes de copiarlas a la presentación.

In [ ]:
print("REPRESENTACIÓN EN EL PDB")
print(f"La búsqueda por {PFAM_ID} recuperó {numero_estructuras} entradas PDB analizables. "
      "Esto describe la disponibilidad de estructuras experimentales anotadas con esta familia, "
      "no la diversidad real de PRAI en la naturaleza.")
print("\nOBSERVACIONES")
if not conteo_metodos.empty:
    print(f"El método más frecuente fue {conteo_metodos.index[0]} "
          f"({int(conteo_metodos.iloc[0])} estructuras).")
if not conteo_organismos.empty:
    print(f"El organismo de origen más representado fue {conteo_organismos.index[0]} "
          f"({int(conteo_organismos.iloc[0])} estructuras).")
if not por_anio.empty:
    anios_maximos = [str(anio) for anio in por_anio[por_anio == por_anio.max()].index]
    print(f"Los años con más depósitos fueron {', '.join(anios_maximos)} "
          f"({int(por_anio.max())} estructuras por año); "
          "estos máximos no permiten inferir tendencias generales con una muestra pequeña.")
print("\nLIMITACIONES")
print("La muestra depende del criterio Pfam y de la cobertura de anotaciones del PDB; "
      "varias estructuras pueden corresponder a la misma proteína, sus mutantes o diferentes ligandos; "
      "existe sesgo hacia organismos y métodos más estudiados; los componentes de cristalización "
      "no necesariamente reflejan ligandos biológicos; y la fecha de depósito no es la fecha de descubrimiento.")

## Archivos reproducibles

Se guardan las tablas y figuras para agregarlas al repositorio de GitHub. También conviene subir este notebook con todas las celdas ejecutadas para que se vean las salidas y gráficas.

In [ ]:
exportar_entradas = entradas.copy()
exportar_entradas["metodos"] = exportar_entradas["metodos"].apply(lambda x: "; ".join(x))
exportar_entradas.to_csv(SALIDA / "entradas_prai.csv", index=False)
entidades.to_csv(SALIDA / "entidades_prai.csv", index=False)
organismos.to_csv(SALIDA / "organismos_prai.csv", index=False)
ligandos.to_csv(SALIDA / "ligandos_prai.csv", index=False)
(SALIDA / "ids_analizados.txt").write_text("\n".join(entradas["pdb_id"]) + "\n", encoding="utf-8")
print("Archivos creados en:", SALIDA.resolve())
for archivo in sorted(SALIDA.rglob("*")):
    if archivo.is_file():
        print(" -", archivo)